# Showcase + Interactive Demo

Dieses Notebook kombiniert zwei Modi:

- eine klare Vorher/Nachher-Showcase-Ansicht
- einen interaktiven Bereich zum Durchklicken und Filtern

Die Daten werden bevorzugt aus `text_bilder/` geladen und fuer die aktuelle Testumgebung ersatzweise aus `test_bilder/`.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import pandas as pd
from PIL import Image

try:
    import ipywidgets as widgets
    from IPython.display import display, Markdown
except ImportError as exc:
    raise ImportError('ipywidgets is required for this notebook. Please ensure the notebook.link environment includes ipywidgets.') from exc


In [ ]:
CANDIDATE_DATASETS = [Path('text_bilder'), Path('test_bilder')]

dataset_root = next((path for path in CANDIDATE_DATASETS if path.exists() and (path / 'bilder').exists()), None)
if dataset_root is None:
    raise FileNotFoundError('Kein Datensatz gefunden. Erwartet wird text_bilder/ oder test_bilder/.')

images_dir = dataset_root / 'bilder'
boxed_images_dir = dataset_root / 'bilder_mit_boxen'
labels_dir = dataset_root / 'bounding_boxen'

print('Datensatz:', dataset_root)
print('Bilder:', images_dir)
print('Labels:', labels_dir)


In [ ]:
def list_image_files(folder: Path):
    patterns = ('*.jpg', '*.jpeg', '*.png', '*.webp')
    files = []
    for pattern in patterns:
        files.extend(folder.glob(pattern))
    return sorted(files)


def parse_yolo_boxes(label_path: Path):
    rows = []
    if not label_path.exists():
        return rows

    for line_no, raw_line in enumerate(label_path.read_text().splitlines(), start=1):
        line = raw_line.strip()
        if not line:
            continue

        parts = line.split()
        if len(parts) != 5:
            raise ValueError(f'Ungueltiges Label-Format in {label_path} Zeile {line_no}: {raw_line!r}')

        class_id, x_center, y_center, width, height = parts
        rows.append({
            'class_id': int(float(class_id)),
            'x_center_norm': float(x_center),
            'y_center_norm': float(y_center),
            'width_norm': float(width),
            'height_norm': float(height),
        })

    return rows


def find_matching_file(folder: Path, stem: str):
    if not folder.exists():
        return None
    for extension in ('.jpg', '.jpeg', '.png', '.webp'):
        candidate = folder / f'{stem}{extension}'
        if candidate.exists():
            return candidate
    return None


def load_image_metadata(dataset_root: Path):
    image_rows = []
    box_rows = []

    for image_path in list_image_files(dataset_root / 'bilder'):
        label_path = dataset_root / 'bounding_boxen' / f'{image_path.stem}.txt'
        boxed_image_path = find_matching_file(dataset_root / 'bilder_mit_boxen', image_path.stem)
        image = Image.open(image_path)
        width, height = image.size
        boxes = parse_yolo_boxes(label_path)

        image_rows.append({
            'stem': image_path.stem,
            'image_name': image_path.name,
            'image_path': str(image_path),
            'label_path': str(label_path) if label_path.exists() else '',
            'boxed_image_path': str(boxed_image_path) if boxed_image_path else '',
            'width_px': width,
            'height_px': height,
            'box_count': len(boxes),
            'has_reference': boxed_image_path is not None,
        })

        for idx, box in enumerate(boxes, start=1):
            width_px = box['width_norm'] * width
            height_px = box['height_norm'] * height
            x_min = (box['x_center_norm'] * width) - (width_px / 2)
            y_min = (box['y_center_norm'] * height) - (height_px / 2)
            area_px = width_px * height_px

            box_rows.append({
                'stem': image_path.stem,
                'box_id': idx,
                'class_id': box['class_id'],
                'x_center_norm': box['x_center_norm'],
                'y_center_norm': box['y_center_norm'],
                'width_norm': box['width_norm'],
                'height_norm': box['height_norm'],
                'x_min_px': x_min,
                'y_min_px': y_min,
                'width_px': width_px,
                'height_px': height_px,
                'area_px': area_px,
            })

    return pd.DataFrame(image_rows), pd.DataFrame(box_rows)


images_df, boxes_df = load_image_metadata(dataset_root)
if images_df.empty:
    raise ValueError(f'Keine Bilder in {images_dir} gefunden.')

images_df = images_df.sort_values('image_name').reset_index(drop=True)
boxes_df = boxes_df.sort_values(['stem', 'box_id']).reset_index(drop=True)


## Showcase-Uebersicht

Diese Kennzahlen sind fuer einen schnellen QR- oder Messestand-Einstieg gedacht.


In [ ]:
total_images = int(len(images_df))
total_boxes = int(len(boxes_df))
images_with_labels = int((images_df['box_count'] > 0).sum())
avg_boxes = round(total_boxes / total_images, 2) if total_images else 0
class_counts = boxes_df['class_id'].value_counts().sort_index() if not boxes_df.empty else pd.Series(dtype='int64')

summary_df = pd.DataFrame([
    {'metric': 'Images', 'value': total_images},
    {'metric': 'Images with boxes', 'value': images_with_labels},
    {'metric': 'Total boxes', 'value': total_boxes},
    {'metric': 'Average boxes per image', 'value': avg_boxes},
])

summary_df

In [ ]:
if class_counts.empty:
    print('Keine Klassenstatistik verfuegbar, weil keine Bounding Boxes gefunden wurden.')
else:
    class_counts.rename_axis('class_id').reset_index(name='count')


## Vorher / Nachher

Die erste Ansicht zeigt ein Beispielbild in drei Varianten: Original, selbst gerenderte Bounding Boxes und optional das vorbereitete Referenzbild mit Boxen.


In [ ]:
def render_showcase(stem: str, selected_classes=None, min_area_px: float = 0.0, show_labels: bool = True, box_color: str = 'lime', line_width: float = 2.0):
    image_row = images_df.loc[images_df['stem'] == stem].iloc[0]
    image = Image.open(image_row['image_path'])
    filtered_boxes = boxes_df.loc[boxes_df['stem'] == stem].copy()

    if selected_classes is not None and len(selected_classes) > 0:
        filtered_boxes = filtered_boxes[filtered_boxes['class_id'].isin(selected_classes)]

    filtered_boxes = filtered_boxes[filtered_boxes['area_px'] >= float(min_area_px)]

    has_reference = bool(image_row['boxed_image_path'])
    fig_cols = 3 if has_reference else 2
    fig, axes = plt.subplots(1, fig_cols, figsize=(6 * fig_cols, 6))

    axes[0].imshow(image)
    axes[0].set_title(f'Original: {image_row['image_name']}')
    axes[0].axis('off')

    axes[1].imshow(image)
    for _, box in filtered_boxes.iterrows():
        rect = Rectangle(
            (box['x_min_px'], box['y_min_px']),
            box['width_px'],
            box['height_px'],
            linewidth=float(line_width),
            edgecolor=box_color,
            facecolor='none'
        )
        axes[1].add_patch(rect)
        if show_labels:
            axes[1].text(
                box['x_min_px'],
                max(0, box['y_min_px'] - 4),
                f"class {int(box['class_id'])}",
                color='black',
                fontsize=8,
                bbox={'facecolor': box_color, 'edgecolor': 'none', 'pad': 1}
            )
    axes[1].set_title(f'Rendered boxes: {len(filtered_boxes)}')
    axes[1].axis('off')

    if has_reference:
        reference_image = Image.open(image_row['boxed_image_path'])
        axes[2].imshow(reference_image)
        axes[2].set_title('Reference image with boxes')
        axes[2].axis('off')

    plt.tight_layout()
    plt.show()

    return filtered_boxes[['box_id', 'class_id', 'x_center_norm', 'y_center_norm', 'width_norm', 'height_norm', 'area_px']].reset_index(drop=True)


default_stem = images_df.iloc[0]['stem']
render_showcase(default_stem)

## Interaktive Steuerung

Hier kann direkt durch die Daten navigiert und die Darstellung angepasst werden.


In [ ]:
available_stems = images_df['stem'].tolist()
available_classes = sorted(boxes_df['class_id'].unique().tolist()) if not boxes_df.empty else []

stem_widget = widgets.Dropdown(options=available_stems, value=default_stem, description='Bild')
classes_widget = widgets.SelectMultiple(options=available_classes, value=tuple(available_classes), description='Klassen')
min_area_widget = widgets.FloatSlider(value=0.0, min=0.0, max=float(boxes_df['area_px'].max()) if not boxes_df.empty else 1.0, step=10.0, description='Min Flaeche')
line_width_widget = widgets.FloatSlider(value=2.0, min=1.0, max=6.0, step=0.5, description='Linienstaerke')
show_labels_widget = widgets.Checkbox(value=True, description='Klassenlabels zeigen')
color_widget = widgets.Dropdown(options=['lime', 'cyan', 'yellow', 'red', 'white'], value='lime', description='Farbe')

controls = widgets.VBox([
    stem_widget,
    classes_widget,
    min_area_widget,
    line_width_widget,
    show_labels_widget,
    color_widget,
])


def update_demo(stem, selected_classes, min_area_px, line_width, show_labels, box_color):
    filtered_df = render_showcase(
        stem=stem,
        selected_classes=list(selected_classes),
        min_area_px=min_area_px,
        show_labels=show_labels,
        box_color=box_color,
        line_width=line_width,
    )
    display(filtered_df)


interactive_output = widgets.interactive_output(
    update_demo,
    {
        'stem': stem_widget,
        'selected_classes': classes_widget,
        'min_area_px': min_area_widget,
        'line_width': line_width_widget,
        'show_labels': show_labels_widget,
        'box_color': color_widget,
    },
)

display(widgets.HBox([controls, interactive_output]))


## Bildtabelle fuer schnellen Ueberblick

Diese Tabelle eignet sich gut, um schnell interessante Kandidaten fuer die Showcase-Ansicht zu finden.


In [ ]:
images_df[['image_name', 'stem', 'width_px', 'height_px', 'box_count', 'has_reference']]
